## Visitor

---

> **In one line.** A visitor separates an *operation* from the *structure* it runs on. Each node carries no operation logic of its own; it exposes a single `accept(e)`, and **double dispatch** routes the call to the right `visit` method on the visitor $e$ — so adding a new operation $V(e, \cdot)$ is exactly one new class and the structure $\mathcal{N}$ never changes.

### 1. The two universes

A visitor design lives between two finite sets. On one side is the **structure**: a set of node types $\mathcal{N}$ — for an expression tree, $\mathcal{N} = \{\text{Number}, \text{Add}, \text{Multiply}\}$. A specific element $n \in \mathcal{N}$ is a single node being visited, and we write $T(n)$ for its **type** — the discriminator used to pick the correct visitor method, e.g. $T(n) = \text{Number}$ selects `visit_number`.

On the other side is the **operation space**: a set of visitor types $\mathcal{E}$, each one a distinct operation over the structure — e.g. $\mathcal{E} = \{\text{EvalVisitor}, \text{PrintVisitor}, \text{TypeCheckVisitor}\}$. A specific $e \in \mathcal{E}$ is one operation to be applied. Every such operation produces a value in a **result type** $Y$: a number for `EvalVisitor`, a string for `PrintVisitor`, and so on.

### 2. What a visitor is

A visitor is the function that takes one operation and one node and returns a result. Pairing the operation space with the structure, the **visitor function** $V$ is the map

$$\boxed{\,V : \mathcal{E} \times \mathcal{N} \longrightarrow Y\,}$$

so that $V(e, n)$ reads as "apply operation $e$ to node $n$." The whole pattern exists to realize this single map *without* writing a method per node type on every operation. It does so through **double dispatch** — two successive dynamic selections that together pin down which concrete code runs:

$$n.\text{accept}(e) \;\;\Longrightarrow\;\; e.\text{visit}_{T(n)}(n) \qquad \text{(double dispatch)}$$

The first dispatch resolves $\text{accept}$ on the dynamic type $T(n)$ of the node; the second resolves $\text{visit}_{T(n)}$ on **both** the visitor $e$ and the node type $T(n)$. Read as a data-flow chain, the call leaves the structure, lands on the matching visitor method, and returns a value:

$$\underbrace{n}_{\mathcal{N}} \;\xrightarrow{\;\text{accept}(e)\;}\; \underbrace{T(n)}_{\text{node type}} \;\xrightarrow{\;\text{visit}_{T(n)}\;}\; \underbrace{e \times n}_{\mathcal{E}\,\times\,\mathcal{N}} \;\longrightarrow\; \underbrace{V(e,n)}_{Y}$$

Because the selection depends on the pair $(e, T(n))$ rather than on either argument alone, neither $\mathcal{N}$ nor $\mathcal{E}$ has to know the concrete identity of the other in advance.

### 3. Key conditions

1. **Structure is fixed.** The nodes in $\mathcal{N}$ never change when new operations $e \in \mathcal{E}$ are added. Adding a new visitor costs exactly one new class:
   $$\mathcal{E} \;\mapsto\; \mathcal{E} \cup \{e_{\text{new}}\}, \qquad \mathcal{N} \;\text{unchanged.}$$
2. **Double dispatch.** The first dispatch selects $\text{accept}()$ based on $T(n)$; the second selects $\text{visit}_{T(n)}$ based on *both* the visitor and the node type. One resolution alone is insufficient — only the pair $(e, T(n))$ determines $V(e,n)$.
3. **Inverse of open–closed.** Adding new operations is cheap (a new visitor class), but adding new node types is expensive (every existing visitor must be updated). Choose the visitor when operations vary more than types.

&nbsp;

> 🏛️ A tax inspector ($e \in \mathcal{E}$) visiting properties ($n \in \mathcal{N}$). Each property simply says "here I am" (`accept`); the inspector knows what to do with each type. Adding a new tax calculation is one new inspector class, and the properties themselves never change.

### Exercise 19 — AST Expression Evaluator

---

**Scenario:** $\mathcal{N} = \{\text{Number, Add, Multiply}\}$. $\mathcal{E} = \{\text{EvalVisitor, PrintVisitor}\}$. Two different operations $V(\cdot, \cdot)$ over the same fixed structure.

**Your task:** Build the tree and both visitors. Adding a `TypeCheckVisitor` should require zero changes to $\mathcal{N}$.

```python
tree = Multiply(Number(3), Add(Number(4), Number(5)))
tree.accept(EvalVisitor())    # -> 27   (double dispatch on each node)
tree.accept(PrintVisitor())   # -> '(3 * (4 + 5))'
```

**Hints**

- Each node $n \in \mathcal{N}$ implements `accept(self, e)` that calls back the matching method on the visitor — this is the *first* dispatch on $T(n)$: `Number.accept` calls `e.visit_number(self)`, `Add.accept` calls `e.visit_add(self)`, etc.
- Each visitor $e \in \mathcal{E}$ implements one method per node type: `visit_number`, `visit_add`, `visit_multiply` — this is the *second* dispatch, selecting $\text{visit}_{T(n)}$ on both the visitor and the node.
- `EvalVisitor` returns a number ($Y = \mathbb{R}$); for `Add`/`Multiply` it recurses via `node.left.accept(self)` and `node.right.accept(self)`. `PrintVisitor` returns a string ($Y = \text{str}$).
- A new operation is one new class: define `TypeCheckVisitor` later with the same three `visit_*` methods — $\mathcal{N}$ stays untouched.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Structure (the fixed set N) — each node implements accept(e): first dispatch on T(n)

class Node(ABC):
    @abstractmethod
    def accept(self, e): ...                 # n.accept(e) => e.visit_{T(n)}(n)

class Number(Node):
    def __init__(self, value):
        self.value = value
    def accept(self, e):
        return e.visit_number(self)          # selects visit_number on visitor e

class Add(Node):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def accept(self, e):
        return e.visit_add(self)             # selects visit_add on visitor e

class Multiply(Node):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def accept(self, e):
        return e.visit_multiply(self)        # selects visit_multiply on visitor e

# --------------------------------
# Visitor interface (the set E) — one visit_* per node type in N

class Visitor(ABC):
    @abstractmethod
    def visit_number(self, n): ...
    @abstractmethod
    def visit_add(self, n): ...
    @abstractmethod
    def visit_multiply(self, n): ...

# --------------------------------
# EvalVisitor (e in E) — your task: V(EvalVisitor, n) -> number (Y = R)

class EvalVisitor(Visitor):
    def visit_number(self, n):
        ...                                  # return the literal value
    def visit_add(self, n):
        ...                                  # recurse: n.left.accept(self) + n.right.accept(self)
    def visit_multiply(self, n):
        ...                                  # recurse: left * right via accept(self)

# --------------------------------
# PrintVisitor (e in E) — your task: V(PrintVisitor, n) -> string (Y = str)

class PrintVisitor(Visitor):
    def visit_number(self, n):
        ...                                  # return str(n.value)
    def visit_add(self, n):
        ...                                  # return f"({left} + {right})" via accept(self)
    def visit_multiply(self, n):
        ...                                  # return f"({left} * {right})" via accept(self)

# --------------------------------
tree = Multiply(Number(3), Add(Number(4), Number(5)))
print("eval :", tree.accept(EvalVisitor()))    # expect 27
print("print:", tree.accept(PrintVisitor()))   # expect (3 * (4 + 5))